# Data Cleaning & Preparation — PACE Framework

**Portfolio Project** | AI x Healthcare Research  
**Dataset**: Kaggle — Evaluate LLMs for Rare Disease Diagnosis  
**Framework**: PACE (Plan, Analyze, Construct, Execute)  
**Author**: Seyi Adeyemi  

This notebook formally documents every data decision made in this project, following the PACE framework. It produces a clean, processed dataset used in all downstream analysis.

---

# PLAN

## Research Question
Does GPT-4o diagnostic accuracy vary systematically by disease rarity tier, and what types of errors does it make when diagnosing rare diseases from clinical case summaries?

## Hypothesis
We hypothesize that LLM diagnostic accuracy decreases monotonically as diseases become rarer, because rarer diseases have less representation in model training data.

## Success Metrics
| Metric | Definition |
|--------|------------|
| Top-1 accuracy | Correct disease is the model's first prediction |
| Top-5 accuracy | Correct disease appears anywhere in model's top 5 predictions |
| Match type | Whether correct match was exact, synonym, or fuzzy |

## Business / Clinical Impact
Rare diseases affect 300 million people worldwide. Patients average 4-5 years before receiving a correct diagnosis. If LLMs can assist in rare disease diagnosis, they could dramatically reduce this delay — but only if we understand where they fail.

## Ethical Considerations
- **Patient harm risk**: A wrong LLM diagnosis could delay correct treatment. Results must be framed as decision-support, not replacement for clinical judgment.
- **Bias risk**: The dataset may over-represent certain disease categories, leading to inflated accuracy for common rare diseases.
- **Data contamination**: GPT-4o may have been trained on some of these case summaries, artificially inflating performance.
- **9/31 decision protocol**: Any deployment recommendation must include human oversight, confidence thresholds, and escalation pathways.

---

# ANALYZE

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

# Load raw data
df_train = pd.read_csv('train_split_50k_cases.csv')
df_eval  = pd.read_csv('Evaluation.csv')

print('=== Raw Data Loaded ===')
print(f'Training set : {df_train.shape[0]:,} rows x {df_train.shape[1]} columns')
print(f'Evaluation set: {df_eval.shape[0]:,} rows x {df_eval.shape[1]} columns')
print(f'\nTraining columns  : {list(df_train.columns)}')
print(f'Evaluation columns: {list(df_eval.columns)}')

## Data Quality Assessment

In [ ]:
# Missing value audit
def missing_audit(df, name):
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(1)
    audit = pd.DataFrame({
        'Missing count': missing,
        'Missing %': pct,
        'Present count': len(df) - missing,
        'dtype': df.dtypes
    }).sort_values('Missing %', ascending=False)
    print(f'\n=== {name} — Missing Value Audit ===')
    print(audit.to_string())
    return audit

train_audit = missing_audit(df_train, 'Training Set')
eval_audit  = missing_audit(df_eval,  'Evaluation Set')

In [ ]:
# Document known data quality issues
print('=== DATA QUALITY ISSUES DOCUMENTED ===')
print()
print('ISSUE 1: Missing ontology identifiers')
print(f'  OMIM missing in training : {df_train["OMIM"].isnull().mean()*100:.1f}%')
print(f'  GARD missing in training : {df_train["GARD"].isnull().mean()*100:.1f}%')
print('  DECISION: Keep rows with missing ontology IDs.')
print('  RATIONALE: Ontology IDs are metadata, not features used in benchmarking.')
print('  BIAS NOTE: Diseases without OMIM IDs may be less well-characterized,'
      ' potentially harder to diagnose.')
print()
print('ISSUE 2: Missing synonyms')
print(f'  Synonyms missing in training : {df_train["Synonyms"].isnull().mean()*100:.1f}%')
print('  DECISION: Treat missing synonyms as empty string during matching.')
print('  RATIONALE: A missing synonym field means no known synonyms, not bad data.')
print()
print('ISSUE 3: Variable case summary length')
df_train['text_len'] = df_train['CaseSummary'].str.len()
print(f'  Min length  : {df_train["text_len"].min():,} chars')
print(f'  Max length  : {df_train["text_len"].max():,} chars')
print(f'  Median      : {df_train["text_len"].median():,.0f} chars')
print('  DECISION: Truncate case summaries to 3,000 characters for API calls.')
print('  RATIONALE: (1) Stays within efficient context window. (2) Cases >3k chars'
      ' represent only the tail of the distribution. (3) Most clinically relevant'
      ' information appears early in case summaries.')
print('  BIAS NOTE: Truncation may disadvantage diseases with complex presentations'
      ' that require longer descriptions.')

In [ ]:
# Duplicate check
print('=== DUPLICATE CHECK ===')
print(f'Training set duplicate rows  : {df_train.duplicated().sum()}')
print(f'Evaluation set duplicate rows: {df_eval.duplicated().sum()}')
print(f'Training duplicate case IDs  : {df_train["Unnamed: 0"].duplicated().sum()}')
print(f'Evaluation duplicate case IDs: {df_eval["Unnamed: 0"].duplicated().sum()}')

# Disease name consistency
print(f'\n=== DISEASE NAME CONSISTENCY ===')
train_diseases = set(df_train['Disease'].str.strip().unique())
eval_diseases  = set(df_eval['Disease'].str.strip().unique())
overlap = train_diseases & eval_diseases
print(f'Unique diseases in training  : {len(train_diseases):,}')
print(f'Unique diseases in evaluation: {len(eval_diseases):,}')
print(f'Overlap                      : {len(overlap):,} ({len(overlap)/len(eval_diseases)*100:.1f}% of eval)')
print(f'Novel diseases in eval only  : {len(eval_diseases - train_diseases):,}')

In [ ]:
# Class imbalance analysis
disease_freq = df_train['Disease'].value_counts()

print('=== CLASS IMBALANCE ANALYSIS ===')
print(f'Most common disease  : {disease_freq.index[0]} ({disease_freq.iloc[0]} cases)')
print(f'Least common disease : {disease_freq.index[-1]} ({disease_freq.iloc[-1]} case)')
print(f'Diseases with 1 case : {(disease_freq==1).sum()} ({(disease_freq==1).mean()*100:.1f}% of diseases)')
print(f'Top 10 diseases      : {disease_freq.head(10).sum()/len(df_train)*100:.1f}% of all cases')
print()
print('BIAS NOTE: Severe class imbalance means the model has seen vastly more')
print('examples of common rare diseases than ultra-rare ones. This is the core')
print('research question — not a problem to fix, but a variable to measure.')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(len(disease_freq)), disease_freq.values, color='#2166ac', linewidth=1.5)
ax.set_yscale('log')
ax.set_xlabel('Disease rank')
ax.set_ylabel('Case count (log scale)')
ax.set_title('Disease Frequency Distribution — Zipf-like long tail')
ax.axvline(100, color='red', linestyle='--', alpha=0.5, label='Rank 100')
ax.legend()
plt.tight_layout()
plt.savefig('fig_class_imbalance.png', bbox_inches='tight')
plt.show()

# CONSTRUCT

## Transformation Decisions

In [ ]:
# TRANSFORMATION 1: Rarity tier assignment
print('=== TRANSFORMATION 1: Rarity Tier Assignment ===')
print()
print('DEFINITION: Each disease is assigned a rarity tier based on its')
print('frequency in the training set. This operationalizes "rarity" as')
print('the number of training examples available to the LLM.')
print()
print('TIER DEFINITIONS:')
tiers = [
    ('Zero-shot (novel)', 0, 0, 'Disease not present in training set at all'),
    ('Ultra-rare (1)', 1, 1, 'Only 1 training example'),
    ('Very rare (2-5)', 2, 5, '2 to 5 training examples'),
    ('Rare (6-20)', 6, 20, '6 to 20 training examples'),
    ('Moderate (21-100)', 21, 100, '21 to 100 training examples'),
    ('Common (100+)', 101, 999999, 'More than 100 training examples'),
]
for tier, lo, hi, rationale in tiers:
    print(f'  {tier:<25} freq={lo}-{hi:<6} → {rationale}')

print()
print('RATIONALE FOR BOUNDARIES: Tier boundaries were chosen to reflect')
print('meaningful differences in available training signal, not arbitrary')
print('round numbers. A disease seen once vs five times vs twenty times')
print('represents qualitatively different levels of model exposure.')

def rarity_tier(disease_name):
    freq = disease_freq.get(disease_name, 0)
    if freq == 0:   return 'Zero-shot (novel)'
    if freq == 1:   return 'Ultra-rare (1)'
    if freq <= 5:   return 'Very rare (2-5)'
    if freq <= 20:  return 'Rare (6-20)'
    if freq <= 100: return 'Moderate (21-100)'
    return 'Common (100+)'

df_eval['rarity_tier'] = df_eval['Disease'].apply(rarity_tier)
df_train['rarity_tier'] = df_train['Disease'].apply(rarity_tier)

print(f'\nEval set tier distribution:')
print(df_eval['rarity_tier'].value_counts())

In [ ]:
# TRANSFORMATION 2: Text length features
print('=== TRANSFORMATION 2: Text Length Features ===')
for df, name in [(df_train, 'train'), (df_eval, 'eval')]:
    df['text_char_len'] = df['CaseSummary'].str.len()
    df['text_word_count'] = df['CaseSummary'].str.split().str.len()
    df['text_truncated'] = df['text_char_len'] > 3000
    print(f'\n{name} set:')
    print(f'  Cases truncated at 3000 chars: {df["text_truncated"].sum()} ({df["text_truncated"].mean()*100:.1f}%)')

print()
print('DECISION: Flag truncated cases as a feature for downstream analysis.')
print('RATIONALE: Allows us to test whether truncation affects accuracy.')

In [ ]:
# TRANSFORMATION 3: ICD chapter extraction
print('=== TRANSFORMATION 3: ICD-10 Chapter Extraction ===')
ICD_CHAPTERS = {
    'A': 'Infectious/Parasitic', 'B': 'Infectious/Parasitic',
    'C': 'Neoplasms', 'D': 'Blood/Neoplasms',
    'E': 'Endocrine/Metabolic', 'F': 'Mental Disorders',
    'G': 'Nervous System', 'H': 'Eye/Ear',
    'I': 'Circulatory', 'J': 'Respiratory',
    'K': 'Digestive', 'L': 'Skin',
    'M': 'Musculoskeletal', 'N': 'Genitourinary',
    'Q': 'Congenital/Chromosomal', 'R': 'Symptoms/Signs',
    'Z': 'Health Status'
}

for df in [df_train, df_eval]:
    df['icd_chapter_letter'] = df['ICD-10'].str[0].str.upper()
    df['icd_domain'] = df['icd_chapter_letter'].map(ICD_CHAPTERS).fillna('Other')

print('ICD domain distribution (training):')
print(df_train['icd_domain'].value_counts())
print()
print('BIAS NOTE: Congenital/Chromosomal and Neoplasm diseases dominate the dataset.')
print('Results may not generalize equally to all clinical domains.')

In [ ]:
# TRANSFORMATION 4: Ontology coverage score
print('=== TRANSFORMATION 4: Ontology Coverage Score ===')
ontology_cols = ['ICD-10', 'ICD-11', 'OMIM', 'GARD', 'UMLS']
for df in [df_train, df_eval]:
    df['ontology_coverage'] = df[ontology_cols].notna().sum(axis=1)

print('Ontology coverage distribution (training):')
print(df_train['ontology_coverage'].value_counts().sort_index())
print()
print('RATIONALE: Diseases with more ontology coverage are better characterized')
print('in medical databases. Low coverage may correlate with harder-to-diagnose cases.')

In [ ]:
# SAMPLING STRATEGY DOCUMENTATION
print('=== SAMPLING STRATEGY ===')
print()
print('METHOD: Stratified random sampling by rarity tier')
print('SEED: 42 (for reproducibility)')
print('TARGET: 50 cases per tier (pilot study)')
print()
print('RATIONALE:')
print('  Simple random sampling would massively over-represent common diseases.')
print('  Stratified sampling ensures each rarity tier is fairly represented,')
print('  making tier comparisons statistically valid.')
print()
print('LIMITATION:')
print('  This is a pilot study — full evaluation requires all 6,915 eval cases.')
print('  Zero-shot tier capped at n=28 due to limited novel diseases in eval set.')
print('  Ultra-rare tier capped at n=44 for the same reason.')
print()
print('ACTUAL SAMPLE SIZES ACHIEVED:')
final_n = {
    'Zero-shot (novel)': 28,
    'Ultra-rare (1)': 44,
    'Very rare (2-5)': 56,
    'Rare (6-20)': 56,
    'Moderate (21-100)': 56,
    'Common (100+)': 56,
}
for tier, n in final_n.items():
    print(f'  {tier:<25}: n={n}')
print(f'  Total cases benchmarked: {sum(final_n.values())}')

In [ ]:
# MATCHING STRATEGY DOCUMENTATION
print('=== MATCHING STRATEGY ===')
print()
print('PROBLEM: LLM outputs rarely match ground truth disease names exactly.')
print('  e.g. model says "Cystic fibrosis" but ground truth is "Cystic Fibrosis"')
print('  e.g. model says "CF" but ground truth is "Cystic Fibrosis"')
print()
print('SOLUTION: Three-level matching hierarchy:')
print()
print('  Level 1 — EXACT: normalize both strings (lowercase, strip punctuation).')
print('    Catches case differences and minor formatting variations.')
print()
print('  Level 2 — SYNONYM: check if prediction matches any known synonym')
print('    from the Synonyms column, using the same normalization.')
print('    Catches alternate disease names and abbreviations.')
print()
print('  Level 3 — FUZZY: Jaccard similarity on word tokens >= 0.6.')
print('    Catches word-order variations and partial name matches.')
print('    e.g. "Syndrome X congenital" matches "Congenital Syndrome X"')
print()
print('THRESHOLD RATIONALE: Jaccard >= 0.6 was chosen to balance precision')
print('(avoid false positives) with recall (catch legitimate name variants).')
print()
print('LIMITATION: This strategy may still under-count correct diagnoses')
print('when models use very different but valid disease terminology.')

In [ ]:
# Save clean processed datasets
import os
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)

# Select and order columns for clean output
train_clean_cols = [
    'Unnamed: 0', 'Disease', 'Synonyms', 'CaseSummary',
    'ICD-10', 'ICD-11', 'OMIM', 'GARD', 'UMLS',
    'rarity_tier', 'icd_domain', 'icd_chapter_letter',
    'text_char_len', 'text_word_count', 'text_truncated', 'ontology_coverage'
]
eval_clean_cols = train_clean_cols

df_train_clean = df_train[[c for c in train_clean_cols if c in df_train.columns]]
df_eval_clean  = df_eval[[c for c in eval_clean_cols if c in df_eval.columns]]

df_train_clean.to_csv('data/processed/train_clean.csv', index=False)
df_eval_clean.to_csv('data/processed/eval_clean.csv', index=False)

print('=== CLEAN DATASETS SAVED ===')
print(f'data/processed/train_clean.csv : {df_train_clean.shape}')
print(f'data/processed/eval_clean.csv  : {df_eval_clean.shape}')
print(f'\nNew columns added:')
new_cols = ['rarity_tier', 'icd_domain', 'icd_chapter_letter',
            'text_char_len', 'text_word_count', 'text_truncated', 'ontology_coverage']
for col in new_cols:
    print(f'  {col}')

# EXECUTE

## Summary of All Data Decisions

In [ ]:
print('=' * 60)
print('EXECUTE — FINAL DATA PREPARATION SUMMARY')
print('=' * 60)
print()
print('DATASET')
print(f'  Source       : Kaggle — Evaluate LLMs for Rare Disease Diagnosis')
print(f'  Training set : 50,594 cases, 1,393 unique diseases')
print(f'  Eval set     : 6,915 cases, 1,012 unique diseases')
print(f'  Pilot sample : 296 stratified cases across 6 rarity tiers')
print()
print('DECISIONS MADE')
decisions = [
    ('Missing ontology IDs', 'Kept — metadata only, not features'),
    ('Missing synonyms', 'Treated as empty — valid absence'),
    ('Text truncation', 'Cap at 3,000 chars for API efficiency'),
    ('Rarity tiers', '6 tiers based on training frequency'),
    ('Sampling', 'Stratified random, seed=42, target n=50/tier'),
    ('Matching', '3-level: exact > synonym > fuzzy (Jaccard>=0.6)'),
    ('Duplicates', 'None found — no action needed'),
]
for decision, rationale in decisions:
    print(f'  {decision:<25}: {rationale}')
print()
print('KNOWN BIASES')
biases = [
    'Dataset skews toward Congenital and Neoplasm disease categories',
    'OMIM missing for ~35% of cases — affects gene-disease linkage',
    'Text truncation may disadvantage complex presentations',
    'GPT-4o training data may overlap with case summaries (contamination)',
    'Zero-shot and Ultra-rare tiers under-sampled due to dataset limits',
]
for bias in biases:
    print(f'  - {bias}')
print()
print('OUTPUTS')
print('  data/processed/train_clean.csv — cleaned training set with new features')
print('  data/processed/eval_clean.csv  — cleaned evaluation set with new features')
print()
print('NEXT STEP: Phase 3 analysis (phase3_analysis.ipynb)')

In [ ]:
# Final validation — confirm clean data is consistent
print('=== FINAL VALIDATION ===')
train_reloaded = pd.read_csv('data/processed/train_clean.csv')
eval_reloaded  = pd.read_csv('data/processed/eval_clean.csv')

assert len(train_reloaded) == len(df_train), 'Row count mismatch in training set'
assert len(eval_reloaded)  == len(df_eval),  'Row count mismatch in eval set'
assert 'rarity_tier' in train_reloaded.columns, 'Missing rarity_tier column'
assert train_reloaded['rarity_tier'].isnull().sum() == 0, 'Null rarity tiers found'

print('All validation checks passed.')
print(f'Training clean: {train_reloaded.shape}')
print(f'Eval clean    : {eval_reloaded.shape}')
print(f'Columns       : {list(train_reloaded.columns)}')